# Lesson 15 | How does the computer talk to a Field-Programmable Gate Array (FPGA)?

Once a board exists, first establish a minimal communication path:

> **How can software write a value to the FPGA and read a result back?**

We call the side running control software the **host**, and the bitstream-configured hardware region **programmable logic (PL)**.

Primary new concept: **the host and PL are separate execution domains.**

## 1. Concept ledger

**Already known:** development board, clock, reset, I/O, bitstream.

**New today:** host, **Central Processing Unit (CPU)**, **System on Chip (SoC)**, and **programmable logic (PL)**.

**Preview only:** a standard on-chip communication protocol is introduced later; channel details are deferred.

## 2. Two execution domains

```mermaid
flowchart LR
  HOST["host software / CPU"] -->|write command| IF["platform control path"]
  IF --> PL["programmable logic register / engine"]
  PL -->|result| IF
  IF -->|readback| HOST
```

Host software executes instructions. PL evolves according to clocks and signals. A register write therefore crosses a hardware communication path.

## 3. Why start with loopback?

A minimal loopback isolates four questions: can the host send, can PL receive/retain/process, can the host read back, and is ordering/state correct?

Do not debug neurons, FIFOs, DDR, and connectome data at the same time.

## 4. Run: model command ordering

This is a **communication-semantics model**, not a real bus implementation. Predict the two readbacks first.

In [ ]:
pl_register = 0
commands = [
    ("write", 7),
    ("add", 5),
    ("read", None),
    ("add", -2),
    ("read", None),
]

readbacks = []
for op, value in commands:
    if op == "write":
        pl_register = value
    elif op == "add":
        pl_register += value
    elif op == "read":
        readbacks.append(pl_register)

print("host readbacks:", readbacks)
print("final PL register:", pl_register)


## 5. Observe

`write` establishes persistent PL state, `add` modifies it, and `read` observes the current value.

The key idea is ordered interaction between host commands and PL state.

## 6. Why is a SoC FPGA convenient?

A SoC tightly integrates processor-system functions and programmable logic. The CPU handles flexible software tasks while PL handles deterministic parallel datapaths.

They remain distinct execution models.

## 7. Try It

Insert `("read", None)` before the first `add`. Predict the new readback order.

## 8. Exercise

[Lesson 15 exercise: model a host ↔ programmable-logic roundtrip](../../exercises/en/15_host_talks_to_fpga.ipynb)

## 9. AI Task

Ask an AI to draw “host write → PL update → host read” and label software versus synchronous-logic actions. Check that it does not confuse function calls with hardware transactions.

## 10. Human Check

Explain what host/CPU versus PL executes, why loopback comes before the neural network, why readback exercises a return path, and why detailed AXI knowledge can wait.

## 11. Engineering Handoff

Maps to `RMD-012B / RMD-013`: complete a minimal host ↔ FPGA loopback, then move the verified small Platform-3 network onto hardware.

## 12. Project Trace

- Lesson: `LSN-015`
- Mapping: `RMD-012B / RMD-013`
- Minimal proof: host write → PL update/state → host read
- Boundary: AXI details deferred

## 13. Exit Ticket

You can explain why host software and FPGA logic are separate execution domains and can trace a minimal roundtrip in order.